In [0]:
from pyspark.sql.functions import col, trim, initcap, upper, when, lit
from delta.tables import DeltaTable

bronze_df = spark.table("ecommerce_ops.bronze.transactions_raw")

# 1. Data Quality Audit: Quarantine invalid records (null IDs or non-positive amounts/quantities)
bad_records_df = bronze_df.filter(
    col("order_id").isNull() |
    col("customer_id").isNull() |
    col("product_id").isNull() |
    (col("quantity") <= 0) |
    (col("total_price_usd") <= 0)
)
bad_records_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce_ops.silver.quarantine_bad_records")

# 2. Keep valid records, standardize text, and deduplicate by order_id
valid_df = (
    bronze_df
    .filter(
        col("order_id").isNotNull() &
        col("customer_id").isNotNull() &
        col("product_id").isNotNull() &
        (col("quantity") > 0) &
        (col("total_price_usd") > 0)
    )
    .dropDuplicates(["order_id"])
    .withColumn("country", initcap(trim(col("country"))))
    .withColumn("city", initcap(trim(col("city"))))
    .withColumn("category", initcap(trim(col("category"))))
    .withColumn("order_status", upper(trim(col("order_status"))))
    .withColumn("payment_status", upper(trim(col("payment_status"))))
)

# Helper function for Incremental Delta MERGE (Upsert)
def upsert_delta_table(source_df, target_table_name, merge_key):
    if spark.catalog.tableExists(target_table_name):
        delta_target = DeltaTable.forName(spark, target_table_name)
        (
            delta_target.alias("target")
            .merge(
                source_df.alias("source"),
                f"target.{merge_key} = source.{merge_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"Incremental MERGE complete on {target_table_name}")
    else:
        source_df.write.format("delta").saveAsTable(target_table_name)
        print(f"Initial load complete on {target_table_name}")

# 3. Normalize & Upsert -> Silver Dimension 1: Customers
customers_df = (
    valid_df
    .select(
        "customer_id", "customer_name", "gender", "age",
        "customer_segment", "country", "city",
        "customer_loyalty_score", "account_creation_date"
    )
    .dropDuplicates(["customer_id"])
)
upsert_delta_table(customers_df, "ecommerce_ops.silver.customers_cleaned", "customer_id")

# 4. Normalize & Upsert -> Silver Dimension 2: Products
products_df = (
    valid_df
    .select(
        "product_id", "product_name", "category", "sub_category",
        "brand", "unit_price_usd", "cost_usd"
    )
    .dropDuplicates(["product_id"])
)
upsert_delta_table(products_df, "ecommerce_ops.silver.products_cleaned", "product_id")

# 5. Normalize & Upsert -> Silver Fact 1: Orders / Transactions
orders_df = valid_df.select(
    "order_id", "customer_id", "product_id", "order_date",
    "order_year", "order_month", "order_status", "quantity",
    "discount_amount_usd", "total_price_usd", "profit_usd",
    "tax_usd", "shipping_cost_usd", "delivery_status"
)
upsert_delta_table(orders_df, "ecommerce_ops.silver.orders_cleaned", "order_id")

# 6. Normalize & Upsert -> Silver Fact 2: Payments & Fraud Risk
payments_df = valid_df.select(
    "order_id", "customer_id", "payment_method", "payment_status",
    "installment_plan", "total_price_usd", "fraud_risk_score"
).withColumn(
    "is_high_fraud_risk",
    when(col("fraud_risk_score") >= 0.75, lit(True)).otherwise(lit(False))
)
upsert_delta_table(payments_df, "ecommerce_ops.silver.payments_cleaned", "order_id")

print("Silver Layer Complete! Clean Orders:", orders_df.count(), "| Quarantined Bad Rows:", bad_records_df.count())


Incremental MERGE complete on ecommerce_ops.silver.customers_cleaned
Incremental MERGE complete on ecommerce_ops.silver.products_cleaned
Incremental MERGE complete on ecommerce_ops.silver.orders_cleaned
Incremental MERGE complete on ecommerce_ops.silver.payments_cleaned
Silver Layer Complete! Clean Orders: 991930 | Quarantined Bad Rows: 0
